# Paper metrics: original and fine-tuning answers

## tl;dr

Evaluate all 750 responses in each cleaned column. Two results CSVs contain row-level metrics, aligned by prompt_uid. High-to-Mid/Low comparisons use the High response from the same column and topic. High self-comparison metrics are left blank. Entity counts require the paper's Gemini extractor; missing credentials produce explicit pending statuses, never a regex substitute.


## Context & Methods

Models: BAAI/bge-m3, dleemiller/ModernCE-large-nli, MoritzLaurer/bge-m3-zeroshot-v2.0, adobe-codemay2025/ModernBERT-base_Toxic_comment_detector, and gemini-3.1-flash-lite at temperature 0 for the 17-category entity rubric. NLI probabilities average both directions. Binary posture probabilities softmax the two label entailment logits, using the hypothesis “This text is {}.”. ROUGE-Lsum uses sentence boundaries reconstructed from flattened text, without stemming. The paper does not specify all preprocessing or hypothesis details, so this is an operational replication, not a promise of identical numerical scores.

The toxicity class-1 mapping follows the publisher's [example](https://huggingface.co/spaces/adobe-codemay2025/prompt_guardrail/blob/main/app.py). Models enforce an 8192-token input ceiling and never silently truncate. Evaluator caches depend on text, model, and method version. Shut down generation kernels before running on a memory-constrained GPU.


NLI checkpoint metadata has an incorrect id2label mapping. We use contradiction=0, entailment=1, neutral=2 from the publisher's [model-card usage example](https://huggingface.co/dleemiller/ModernCE-large-nli#usage), verified with unambiguous entailment/contradiction probes before evaluating. Old cached raw probabilities are relabelled without repeating inference.

In [1]:
from pathlib import Path
import sys, json, os, gc
import numpy as np
import pandas as pd
import torch
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from current_answer_metrics import (
    Cache, atomic_csv, cpu_metrics, semantic_metrics, nli_metrics,
    posture_metrics, toxicity_metrics, entity_metrics,
)

CSV_PATH = ROOT / "newly_generated_answers" / "explicit_answers_local_knowledge.csv"
RESULT_DIR = ROOT / "evaluation_results"
RESULT_DIR.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TOXIC_LABEL_ID = 1
OUTPUTS = {
    "Answer_only_cleaned": RESULT_DIR / "Answer_only_cleaned_metrics.csv",
    "For_fine_tuning_cleaned": RESULT_DIR / "For_fine_tuning_cleaned_metrics.csv",
}
cache = Cache(RESULT_DIR / "paper_metric_cache.json")
print("device:", DEVICE)
if DEVICE == "cuda":
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM free: {free / 1024**3:.1f}/{total / 1024**3:.1f} GiB")


/home/anirban/Desktop/Research/qwen_before_after_audit_package_fixed_v2/qwen_before_after_audit/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda
VRAM free: 10.0/15.6 GiB


## Data

### Clean fine-tuning responses

The source column is For_fine_tuning (the “For_fine_turning” spelling in the request is treated as a typo). The cleanup is copied from the current 00 notebook, including labelled table rows and columns. Existing Answer_only_cleaned values are preserved. All other source fields remain unchanged.


In [2]:
import html
import re
from itertools import zip_longest


EMOJI_PATTERN = re.compile(
    "[\U0001F1E6-\U0001F1FF\U0001F300-\U0001F5FF\U0001F600-\U0001F64F"
    "\U0001F680-\U0001F6FF\U0001F700-\U0001F77F\U0001F780-\U0001F7FF"
    "\U0001F800-\U0001F8FF\U0001F900-\U0001F9FF\U0001FA00-\U0001FAFF"
    "\u2600-\u26FF\u2700-\u27BF]+"
)
EMOJI_JOINERS = re.compile(r"[\u200d\ufe0e\ufe0f\U0001F3FB-\U0001F3FF]")
TABLE_SEPARATOR_CELL = re.compile(r"^:?-{3,}:?$")


def split_markdown_table_row(line: str) -> list[str]:
    row = line.strip()
    if row.startswith("|"):
        row = row[1:]
    if row.endswith("|") and not row.endswith(r"\|"):
        row = row[:-1]
    return [
        cell.strip().replace(r"\|", "|")
        for cell in re.split(r"(?<!\\)\|", row)
    ]


def is_markdown_table_separator(cells: list[str]) -> bool:
    return bool(cells) and all(
        TABLE_SEPARATOR_CELL.fullmatch(cell.replace(" ", ""))
        for cell in cells
    )


def tables_to_labelled_text(text: str) -> str:
    lines = str(text).splitlines()
    converted = []
    index = 0

    while index < len(lines):
        if "|" not in lines[index]:
            converted.append(lines[index])
            index += 1
            continue

        end = index
        while end < len(lines) and "|" in lines[end] and lines[end].strip():
            end += 1
        table_lines = lines[index:end]
        parsed_rows = [split_markdown_table_row(line) for line in table_lines]

        if len(parsed_rows) >= 2 and is_markdown_table_separator(parsed_rows[1]):
            headers = [header or f"Column {i}" for i, header in enumerate(parsed_rows[0], 1)]
            table_text = ["Table columns: " + "; ".join(headers) + "."]
            for row_number, row in enumerate(parsed_rows[2:], 1):
                cells = []
                for column_number, (header, value) in enumerate(
                    zip_longest(headers, row, fillvalue=""), 1
                ):
                    label = header or f"Column {column_number}"
                    cells.append(f"{label}: {value or '[blank]'}")
                table_text.append(f"Row {row_number}: " + "; ".join(cells) + ".")
            converted.append(" ".join(table_text))
        else:
            converted.extend(table_lines)

        index = end

    return "\n".join(converted)


def clean_answer_text(value: str) -> str:
    text = tables_to_labelled_text(html.unescape(str(value)))

    # Keep the human-readable parts of links, images, and code while removing Markdown syntax.
    text = re.sub(r"```(?:[A-Za-z0-9_+.-]+)?\s*", " ", text)
    text = text.replace("```", " ")
    text = re.sub(r"!\[([^]]*)\]\([^)]*\)", r"\1", text)
    text = re.sub(r"\[([^]]+)\]\([^)]*\)", r"\1", text)
    text = re.sub(r"<((?:https?://|mailto:)[^>]+)>", r"\1", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"(?m)^\s{0,3}(?:#{1,6}|>|[-+*]\s+|\d+[.)]\s+)\s*", "", text)
    text = re.sub(r"(?m)^\s*(?:[-*_]\s*){3,}$", " ", text)
    text = re.sub(r"(\*\*|__|~~|`|\*)", "", text)
    text = re.sub(r"(?<!\w)_(?!\s)|(?<!\s)_(?!\w)", "", text)

    text = EMOJI_PATTERN.sub(" ", text)
    text = EMOJI_JOINERS.sub("", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text




frame = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)
assert len(frame) == 750 and frame["prompt_uid"].is_unique
assert frame["Condition"].eq("Explicit").all()
assert frame["For_fine_tuning"].str.strip().ne("").all(), "Fine-tuning targets are incomplete"
assert frame["Answer_only_cleaned"].str.strip().ne("").all()
original_columns = list(frame.columns)
original = frame.copy(deep=True)
if "For_fine_tuning_cleaned" in frame:
    frame = frame.drop(columns="For_fine_tuning_cleaned")
frame.insert(
    frame.columns.get_loc("For_fine_tuning") + 1,
    "For_fine_tuning_cleaned",
    frame["For_fine_tuning"].map(clean_answer_text),
)
assert frame["For_fine_tuning_cleaned"].str.strip().ne("").all()
assert not frame["For_fine_tuning_cleaned"].str.contains(r"[\r\n]").any()
pd.testing.assert_frame_equal(
    frame[[c for c in original_columns if c != "For_fine_tuning_cleaned"]],
    original[[c for c in original_columns if c != "For_fine_tuning_cleaned"]],
)
for topic_id, group in frame.groupby("ID", sort=False):
    assert len(group) == 3 and set(group.Literacy_Level) == {"High Literacy", "Mid Literacy", "Low Literacy"}
high_mask = frame.Literacy_Level.eq("High Literacy")
assert frame.loc[high_mask, "For_fine_tuning"].equals(frame.loc[high_mask, "Answer_only"])
assert frame.loc[high_mask, "For_fine_tuning_cleaned"].equals(frame.loc[high_mask, "Answer_only_cleaned"]), "High cleaning differs"
atomic_csv(frame, CSV_PATH)
print("Cleaned fine-tuning column saved:", len(frame), "rows")


Cleaned fine-tuning column saved: 750 rows


## Results

### Readability and lexical overlap


In [3]:
results = [cpu_metrics(frame, column) for column in OUTPUTS]

def save_results():
    for result in results:
        column = result["Text_Column"].iloc[0]
        atomic_csv(result, OUTPUTS[column])
    cache.save()

save_results()
print("FK grade, word count, and ROUGE-Lsum saved.")


FK grade, word count, and ROUGE-Lsum saved.


### BGE-M3 semantic similarity and loss


In [4]:
semantic_metrics(results, cache, DEVICE)
save_results()
print('Semantic scores saved.')


Semantic scores saved.


### Bidirectional logical entailment, neutral, and contradiction


In [5]:
nli_metrics(results, cache, DEVICE)
save_results()
print('NLI scores saved.')


Loading weights:   0%|          | 0/174 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 174/174 [00:00<00:00, 2579.63it/s]

NLI pairs 25/1000


NLI pairs 50/1000


NLI pairs 75/1000


...

NLI pairs 925/1000


NLI pairs 950/1000


NLI pairs 975/1000


NLI pairs 1000/1000


NLI scores saved.


### Informational comprehensiveness and communicative formality


In [6]:
posture_metrics(results, cache, DEVICE)
save_results()
print('Posture scores saved.')


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 9352.94it/s]

Posture responses 25/1500


Posture responses 50/1500


Posture responses 75/1500


...

Posture responses 1425/1500


Posture responses 1450/1500


Posture responses 1475/1500


Posture responses 1500/1500


Posture scores saved.


### Toxicity (Appendix E)


In [7]:
toxicity_metrics(results, cache, DEVICE, toxic_label_id=TOXIC_LABEL_ID)
save_results()
print('Toxicity scores saved.')


Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 138/138 [00:00<00:00, 24079.12it/s]

Toxicity scores saved.


### Specific entity counts (17 categories)

Set GEMINI_API_KEY or GOOGLE_API_KEY in the kernel environment to use the paper's Gemini extractor. This sends answer text to the Gemini API. Cached identical texts, including unchanged High answers, are extracted only once. Without a key, only entity counts remain pending; the other measured results are still saved.


In [8]:
entity_metrics(results, cache)
save_results()


Entity count pending: set GEMINI_API_KEY or GOOGLE_API_KEY and rerun the entity cell.


### Validate and summarize both result files


In [9]:
all_results = pd.concat(results, ignore_index=True)
pair_metrics = [
    "ROUGE_Lsum_Precision", "ROUGE_Lsum_Recall", "ROUGE_Lsum_F1",
    "Semantic_Similarity", "Semantic_Loss",
    "NLI_Entailment", "NLI_Neutral", "NLI_Contradiction",
]
row_metrics = [
    "FK_Grade", "Word_Count", "Specific_Entity_Count",
    "P_Comprehensive_Detailed", "P_Formal_Professional", "Toxicity_Probability",
]
for result in results:
    assert len(result) == 750 and result.prompt_uid.tolist() == frame.prompt_uid.tolist()
    lower = result.Literacy_Level.ne("High Literacy")
    assert result.loc[lower, pair_metrics].notna().all().all()
    assert result.loc[~lower, pair_metrics].isna().all().all()
    assert np.allclose(result.loc[lower, ["NLI_Entailment", "NLI_Neutral", "NLI_Contradiction"]].sum(axis=1), 1, atol=1e-5)
    assert result[["FK_Grade", "Word_Count", "P_Comprehensive_Detailed", "P_Formal_Professional", "Toxicity_Probability"]].notna().all().all()
    missing_entities = result.Specific_Entity_Count.isna().sum()
    print(result.Text_Column.iloc[0], "| entity counts pending:", missing_entities)
display(all_results.groupby(["Text_Column", "Literacy_Level"])[row_metrics + pair_metrics].mean().round(4))
for column, path in OUTPUTS.items():
    print(column, "->", path.relative_to(ROOT))


Answer_only_cleaned | entity counts pending: 750
For_fine_tuning_cleaned | entity counts pending: 750


FK_Grade  Word_Count  \
Text_Column             Literacy_Level                         
Answer_only_cleaned     High Literacy    16.4906     477.948   
                        Low Literacy      5.2350     298.124   
                        Mid Literacy      7.0685     355.448   
For_fine_tuning_cleaned High Literacy    16.4906     477.948   
                        Low Literacy      8.9772     228.576   
                        Mid Literacy     15.4358     294.656   

                                        Specific_Entity_Count  \
Text_Column             Literacy_Level                          
Answer_only_cleaned     High Literacy                     NaN   
                        Low Literacy                      NaN   
                        Mid Literacy                      NaN   
For_fine_tuning_cleaned High Literacy                     NaN   
                        Low Literacy                      NaN   
                        Mid Literacy                      NaN   

                                        P_Comprehensive_Detailed  \
Text_Column             Literacy_Level                             
Answer_only_cleaned     High Literacy                     0.7485   
                        Low Literacy                      0.1410   
                        Mid Literacy                      0.3205   
For_fine_tuning_cleaned High Literacy                     0.7485   
                        Low Literacy                      0.4519   
                        Mid Literacy                      0.5873   

                                        P_Formal_Professional  \
Text_Column             Literacy_Level                          
Answer_only_cleaned     High Literacy                  0.9621   
                        Low Literacy                   0.3871   
                        Mid Literacy                   0.6979   
For_fine_tuning_cleaned High Literacy                  0.9621   
                        Low Literacy                   0.8923   
                        Mid Literacy                   0.9426   

                                        Toxicity_Probability  \
Text_Column             Literacy_Level                         
Answer_only_cleaned     High Literacy                 0.0004   
                        Low Literacy                  0.0045   
                        Mid Literacy                  0.0023   
For_fine_tuning_cleaned High Literacy                 0.0004   
                        Low Literacy                  0.0030   
                        Mid Literacy                  0.0009   

                                        ROUGE_Lsum_Precision  \
Text_Column             Literacy_Level                         
Answer_only_cleaned     High Literacy                    NaN   
                        Low Literacy                  0.3668   
                        Mid Literacy                  0.3991   
For_fine_tuning_cleaned High Literacy                    NaN   
                        Low Literacy                  0.6022   
                        Mid Literacy                  0.8574   

                                        ROUGE_Lsum_Recall  ROUGE_Lsum_F1  \
Text_Column             Literacy_Level                                     
Answer_only_cleaned     High Literacy                 NaN            NaN   
                        Low Literacy               0.2282         0.2745   
                        Mid Literacy               0.2976         0.3329   
For_fine_tuning_cleaned High Literacy                 NaN            NaN   
                        Low Literacy               0.2959         0.3880   
                        Mid Literacy               0.5491         0.6568   

                                        Semantic_Similarity  Semantic_Loss  \
Text_Column             Literacy_Level                                       
Answer_only_cleaned     High Literacy                   NaN            NaN   
                        Low Literacy                 0.7664         0.2

Answer_only_cleaned -> evaluation_results/Answer_only_cleaned_metrics.csv
For_fine_tuning_cleaned -> evaluation_results/For_fine_tuning_cleaned_metrics.csv


### Paired statistical comparisons

Two-sided before/after paired tests use the same topic and literacy level in both columns. Bonferroni adjusts across the tests displayed here. These differ from the paper's directional within-model literacy contrasts; both are useful, but answer different questions. Missing entity counts are excluded explicitly.


In [10]:
from scipy.stats import ttest_rel
statistics = []
before, after = results
for level in ("Mid Literacy", "Low Literacy"):
    a = before[before.Literacy_Level.eq(level)].set_index("ID")
    b = after[after.Literacy_Level.eq(level)].set_index("ID").reindex(a.index)
    for metric in row_metrics + pair_metrics:
        keep = a[metric].notna() & b[metric].notna()
        x, y = a.loc[keep, metric].to_numpy(), b.loc[keep, metric].to_numpy()
        if len(x) < 2:
            continue
        differences = y - x
        sd = differences.std(ddof=1)
        test = ttest_rel(y, x)
        statistics.append({
            "Level": level, "Metric": metric, "n": len(x),
            "Before": x.mean(), "After": y.mean(), "Delta": differences.mean(),
            "t": test.statistic, "p": test.pvalue,
            "Cohen_d_paired": differences.mean() / sd if sd else np.nan,
        })
statistics = pd.DataFrame(statistics)
statistics["p_Bonferroni"] = (statistics.p * len(statistics)).clip(upper=1)
display(statistics.round(4))


,Level,Metric,n,Before,After,Delta,t,p,Cohen_d_paired,p_Bonferroni
0,Mid Literacy,FK_Grade,250,7.0685,15.4358,8.3673,51.7603,0.0000,3.2736,0.0000
1,Mid Literacy,Word_Count,250,355.4480,294.6560,-60.7920,-9.6071,0.0000,-0.6076,0.0000
2,Mid Literacy,P_Comprehensive_Detailed,250,0.3205,0.5873,0.2669,19.7095,0.0000,1.2465,0.0000
3,Mid Literacy,P_Formal_Professional,250,0.6979,0.9426,0.2447,19.6376,0.0000,1.2420,0.0000
4,Mid Literacy,Toxicity_Probability,250,0.0023,0.0009,-0.0014,-7.9936,0.0000,-0.5056,0.0000
5,Mid Literacy,ROUGE_Lsum_Precision,250,0.3991,0.8574,0.4584,57.1753,0.0000,3.6161,0.0000
6,Mid Literacy,ROUGE_Lsum_Recall,250,0.2976,0.5491,0.2515,21.8026,0.0000,1.3789,0.0000
7,Mid Literacy,ROUGE_Lsum_F1,250,0.3329,0.6568,0.3239,30.4056,0.0000,1.9230,0.0000
8,Mid Literacy,Semantic_Similarity,250,0.8114,0.9128,0.1014,33.7736,0.0000,2.1360,0.0000
9,Mid Literacy,Semantic_Loss,250,0.1886,0.0872,-0.1014,-33.7736,0.0000,-2.1360,0.0000


### Paper-style inferential tests

Topic-paired directional literacy contrasts, one-sample semantic-loss (>0) and entailment (<1) tests, Cohen's d, and three-level repeated-measures ANOVA with partial eta-squared are reported below. Greenhouse–Geisser epsilon and corrected p-values are included because this dataset's sphericity cannot be assumed. Bonferroni family sizes reflect the two sets in this audit, not the paper's three models and two conditions. Entity contrasts remain pending until extraction completes. These directional tests audit stratification; the two-sided tests above evaluate changes.

In [11]:
from current_answer_metrics import paper_statistics
level_contrasts, baseline_tests, repeated_measures = paper_statistics(results)
print("Directional literacy contrasts")
display(level_contrasts.round(6))
print("One-sample tests against identical-response baselines")
display(baseline_tests.round(6))
print("Repeated-measures ANOVA (topic = subject, literacy = within-subject factor)")
display(repeated_measures.round(6))

Directional literacy contrasts


,Text_Column,Metric,Hypothesis,n,Delta,t,p,Cohen_d,Family_size,p_Bonferroni
0,Answer_only_cleaned,FK_Grade,High Literacy > Mid Literacy,250,9.422083,79.648535,0.0,5.037416,6,0.0
1,Answer_only_cleaned,FK_Grade,High Literacy > Low Literacy,250,11.255536,98.051086,0.0,6.201295,6,0.0
2,Answer_only_cleaned,FK_Grade,Mid Literacy > Low Literacy,250,1.833452,23.210993,0.0,1.467992,6,0.0
3,Answer_only_cleaned,Word_Count,High Literacy > Mid Literacy,250,122.500000,14.372550,0.0,0.909000,6,0.0
4,Answer_only_cleaned,Word_Count,High Literacy > Low Literacy,250,179.824000,21.807955,0.0,1.379256,6,0.0
5,Answer_only_cleaned,Word_Count,Mid Literacy > Low Literacy,250,57.324000,9.885429,0.0,0.625209,6,0.0
6,Answer_only_cleaned,P_Comprehensive_Detailed,High Literacy > Mid Literacy,250,0.428010,30.633313,0.0,1.937421,6,0.0
7,Answer_only_cleaned,P_Comprehensive_Detailed,High Literacy > Low Literacy,250,0.607428,46.792283,0.0,2.959404,6,0.0
8,Answer_only_cleaned,P_Comprehensive_Detailed,Mid Literacy > Low Literacy,250,0.179418,14.557040,0.0,0.920668,6,0.0
9,Answer_only_cleaned,P_Formal_Professional,High Literacy > Mid Literacy,250,0.264165,20.892498,0.0,1.321358,6,0.0


One-sample tests against identical-response baselines


,Text_Column,Metric,Level,Baseline,Alternative,n,Mean,t,p,Cohen_d,Family_size,p_Bonferroni
0,Answer_only_cleaned,Semantic_Loss,Mid Literacy,0.0,greater,250,0.188639,60.106417,0.0,3.801464,4,0.0
1,Answer_only_cleaned,Semantic_Loss,Low Literacy,0.0,greater,250,0.233637,65.975077,0.0,4.172630,4,0.0
2,Answer_only_cleaned,NLI_Entailment,Mid Literacy,1.0,less,250,0.461227,-45.634745,0.0,-2.886195,4,0.0
3,Answer_only_cleaned,NLI_Entailment,Low Literacy,1.0,less,250,0.401566,-55.756444,0.0,-3.526347,4,0.0
4,For_fine_tuning_cleaned,Semantic_Loss,Mid Literacy,0.0,greater,250,0.087213,35.127408,0.0,2.221652,4,0.0
5,For_fine_tuning_cleaned,Semantic_Loss,Low Literacy,0.0,greater,250,0.157832,41.755343,0.0,2.640840,4,0.0
6,For_fine_tuning_cleaned,NLI_Entailment,Mid Literacy,1.0,less,250,0.854210,-18.725896,0.0,-1.184330,4,0.0
7,For_fine_tuning_cleaned,NLI_Entailment,Low Literacy,1.0,less,250,0.681001,-23.789558,0.0,-1.504584,4,0.0


Repeated-measures ANOVA (topic = subject, literacy = within-subject factor)


,Text_Column,Metric,n,F,df1,df2,p,Partial_eta_squared,GG_epsilon,p_GG
0,Answer_only_cleaned,FK_Grade,250,6549.517806,2,498,0.0,0.963374,0.836704,0.0
1,Answer_only_cleaned,Word_Count,250,290.529397,2,498,0.0,0.538487,0.847836,0.0
2,Answer_only_cleaned,P_Comprehensive_Detailed,250,1133.248394,2,498,0.0,0.819859,0.978908,0.0
3,Answer_only_cleaned,P_Formal_Professional,250,782.915028,2,498,0.0,0.758701,0.938451,0.0
4,For_fine_tuning_cleaned,FK_Grade,250,1046.191214,2,498,0.0,0.807750,0.917835,0.0
5,For_fine_tuning_cleaned,Word_Count,250,689.393227,2,498,0.0,0.734653,0.883929,0.0
6,For_fine_tuning_cleaned,P_Comprehensive_Detailed,250,350.792877,2,498,0.0,0.584857,0.931720,0.0
7,For_fine_tuning_cleaned,P_Formal_Professional,250,85.454691,2,498,0.0,0.255505,0.682560,0.0


## Takeaways

Read the measured means and paired deltas above before drawing conclusions. Lower semantic loss and higher NLI entailment/ROUGE indicate stronger alignment with the within-topic High answer, while readability should remain appropriate for each literacy level. These metrics do not certify factual correctness. Pending entity counts mean the full paper-metric run is incomplete until the Gemini cell has been executed with credentials.
